In [1]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from front_finding import buildconfig
from front_finding.cli import build_fronts
from front_finding.finding import io as finding_io
from front_finding.llc import io as llc_io
from front_finding.llc import source as llc_source
from front_finding.properties import io as properties_io

CONFIG = Path("../configs/run/run_test_single_timestep.yaml")
assert CONFIG.is_file(), f"run this from notebooks/ -- {CONFIG} not found"

cfg = buildconfig.load_config(CONFIG)

TIMESTAMP = cfg.timestamps[0]

# run() sets this itself; doing it here means the paths below resolve before
# anything has been generated.
llc_io.set_fronts_path(cfg.products.root)
llc_io.set_run_layout(cfg.run_dir, file_tag=cfg.run_id)

RUN_ROOT = llc_io.run_root(cfg.run_id)
OUT_DIR = llc_io.fronts_dir(cfg.run_id, TIMESTAMP)
TIME_STR = TIMESTAMP.replace("_", ":")
RUN_TAG = f"{cfg.run_id}_bfronts"       # how group and colocate tag their output

print(f"{cfg.pipeline}  {TIMESTAMP}  finding config {cfg.finding.config}")
print(f"stores   s3://{cfg.source.bucket.strip('/')}/"
      f"{cfg.source.folder.strip('/')}/{cfg.run_id}/")
print(f"grid     s3://{cfg.source.grid['bucket']}/{cfg.source.grid['folder']}/"
      f"{cfg.source.grid['dataset_name']}")
print(f"products {OUT_DIR}")

SURF  2011-12-04T00_00_00  finding config D
stores   s3://dbof/test_globals_for_front_finding/test01/
grid     s3://dbof/LLC4320_GRID_2D/llc4320_grid.zarr
products /Users/jaketallman/PycharmProjects/front-finding/output/TEST01/SURF/20111204_000000


## Run

One call, four steps:

| step | does |
|---|---|
| `gradb2` | build `frontal_structure.zarr` if it is missing (a dbof run) |
| `find` | threshold gradb2 into a binary front map |
| `group` | label the fronts, measure their geometry |
| `colocate` | sample the subset's other 7 channels onto each front |

`frontal_structure` carries `gradb2, gradsalt2, gradtheta2, gradeta2,
gradrho2, turner_angle, density, buoyancy`, so `colocate` costs no extra
generation — every field is already in the store built by the first step, and
is read from it directly.

Drop steps from the list to re-run part of it.

In [ ]:
build_fronts.run(cfg, ["gradb2", "find", "group", "colocate"])

pipeline=SURF  run_id=test01  dates=1  gradb2=gradb2 (subset=frontal_structure)  finding_config=D
steps=['gradb2', 'find', 'group', 'colocate']
products -> /Users/jaketallman/PycharmProjects/front-finding/output/TEST01/SURF
  GENERATE  frontal_structure  (missing)
Running: /Users/jaketallman/PycharmProjects/front-finding/.venv/bin/python -m dbof.cli.run_all_subsets --config /var/folders/kr/y4zmny8x7tlbdrx9d7bkt0l00000gn/T/dbof_source_uc4hz1kj.yaml --netcdf-base /Users/jaketallman/PycharmProjects/front-finding/output/TEST01/SURF --subsets frontal_structure --generate-only


2026-09-01 15:37:46 | INFO | Pipeline: SURF  |  run_id: test01  |  dates: 1  |  subsets: 1
2026-09-01 15:37:46 | INFO |   frontal_structure  (frontal_structure.zarr, 8 channels)
2026-09-01 15:37:46 | INFO | Found credentials in shared credentials file: ~/.aws/credentials
2026-09-01 15:37:46 | INFO | 
2026-09-01 15:37:46 | INFO | WORK PLAN (.nc-first: check .nc -> check zarr -> generate/export)
2026-09-01 15:37:46 | INFO |   frontal_structure      20111204_000000  GENERATE 
2026-09-01 15:37:46 | INFO | 
2026-09-01 15:37:46 | INFO | ============================================================
2026-09-01 15:37:46 | INFO | PHASE 1: GENERATE ZARR STORES
2026-09-01 15:37:46 | INFO | ============================================================
2026-09-01 15:37:46 | INFO | ============================================================
2026-09-01 15:37:46 | INFO | GENERATE  config=/var/folders/kr/y4zmny8x7tlbdrx9d7bkt0l00000gn/T/dbof_source_uc4hz1kj.yaml  subset=frontal_structure  pipeline=SURF
2

2026-09-01 15:37:48,972 | INFO | Log file: /Users/jaketallman/PycharmProjects/front-finding/logs/test01/frontal_structure_20260901_223748.log
2026-09-01 15:37:48,972 | INFO | Unified global pipeline starting.
2026-09-01 15:37:48,972 | INFO | Pipeline: SURF
2026-09-01 15:37:48,972 | INFO | Active subsets: ['frontal_structure']
2026-09-01 15:37:48,972 | INFO | Depth suffixes (YAML override): None
2026-09-01 15:37:48,972 | INFO | Dates: ['2011-12-04 00:00:00']
2026-09-01 15:37:48,972 | INFO | Pre-flight plan (zarr existence per subset/date):
2026-09-01 15:37:49,115 | INFO |   frontal_structure      2011-12-04 00:00:00  ->  GENERATE (zarr store missing)
2026-09-01 15:37:49,450 | INFO | To route to workers diagnostics web server please install jupyter-server-proxy: python -m pip install jupyter-server-proxy
2026-09-01 15:37:49,459 | INFO | State start
2026-09-01 15:37:49,461 | INFO |   Scheduler at:     tcp://127.0.0.1:64184
2026-09-01 15:37:49,461 | INFO |   dashboard at:  http://127.0.0.1

/Users/jaketallman/PycharmProjects/front-finding/.venv/lib/python3.12/site-packages/distributed/node.py:187: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 64183 instead
  warnings.warn(
2026-09-01 15:37:49 | INFO |       Start worker at:      tcp://127.0.0.1:64196
2026-09-01 15:37:49 | INFO |          Listening to:      tcp://127.0.0.1:64196
2026-09-01 15:37:49 | INFO |       Start worker at:      tcp://127.0.0.1:64195
2026-09-01 15:37:49 | INFO |          Listening to:      tcp://127.0.0.1:64195
2026-09-01 15:37:49 | INFO |           Worker name:                          1
2026-09-01 15:37:49 | INFO |          dashboard at:            127.0.0.1:64197
2026-09-01 15:37:49 | INFO | Waiting to connect to:      tcp://127.0.0.1:64184
2026-09-01 15:37:49 | INFO | -------------------------------------------------
2026-09-01 15:37:49 | INFO |               Threads:                          4
2026-09-01 15:37:49 | INFO |   

2026-09-01 15:37:49,763 | INFO | Register worker addr: tcp://127.0.0.1:64195 name: 0
2026-09-01 15:37:49,775 | INFO | Starting worker compute stream, tcp://127.0.0.1:64195
2026-09-01 15:37:49,775 | INFO | Starting established connection to tcp://127.0.0.1:64200
2026-09-01 15:37:49,775 | INFO | Register worker addr: tcp://127.0.0.1:64196 name: 1
2026-09-01 15:37:49,776 | INFO | Starting worker compute stream, tcp://127.0.0.1:64196
2026-09-01 15:37:49,776 | INFO | Starting established connection to tcp://127.0.0.1:64199
2026-09-01 15:37:49,785 | INFO | Register worker addr: tcp://127.0.0.1:64204 name: 3
2026-09-01 15:37:49,786 | INFO | Starting worker compute stream, tcp://127.0.0.1:64204
2026-09-01 15:37:49,786 | INFO | Starting established connection to tcp://127.0.0.1:64206
2026-09-01 15:37:49,786 | INFO | Register worker addr: tcp://127.0.0.1:64201 name: 2
2026-09-01 15:37:49,786 | INFO | Starting worker compute stream, tcp://127.0.0.1:64201
2026-09-01 15:37:49,786 | INFO | Starting 

...

Exception ignored in: <function Client.__del__ at 0x133a48b80>
Traceback (most recent call last):
  File "/Users/jaketallman/PycharmProjects/front-finding/.venv/lib/python3.12/site-packages/distributed/client.py", line 1726, in __del__
    self.close()
  File "/Users/jaketallman/PycharmProjects/front-finding/.venv/lib/python3.12/site-packages/distributed/client.py", line 1981, in close
    sync(self.loop, self._close, fast=True, callback_timeout=timeout)
  File "/Users/jaketallman/PycharmProjects/front-finding/.venv/lib/python3.12/site-packages/distributed/utils.py", line 446, in sync
    raise TimeoutError(f"timed out after {timeout} s.")
TimeoutError: timed out after 60 s.


Wrote run descriptor: /Users/jaketallman/PycharmProjects/front-finding/output/TEST01/SURF/fronts_meta_TEST01_SURF_from_test_globals_for_front_finding_test01.meta
Running: /Users/jaketallman/PycharmProjects/front-finding/.venv/bin/python -m dbof.cli.run_all_subsets --config /var/folders/kr/y4zmny8x7tlbdrx9d7bkt0l00000gn/T/dbof_source_mulgnud3.yaml --netcdf-base /Users/jaketallman/PycharmProjects/front-finding/output/TEST01/SURF --generate-only


2026-09-01 15:49:02 | INFO | Pipeline: SURF  |  run_id: test01  |  dates: 1  |  subsets: 1
2026-09-01 15:49:02 | INFO |   frontal_structure  (frontal_structure.zarr, 8 channels)
2026-09-01 15:49:02 | INFO | Found credentials in shared credentials file: ~/.aws/credentials
2026-09-01 15:49:03 | INFO | 
2026-09-01 15:49:03 | INFO | WORK PLAN (.nc-first: check .nc -> check zarr -> generate/export)
2026-09-01 15:49:03 | INFO |   frontal_structure      20111204_000000  SKIP     
2026-09-01 15:49:03 | INFO | 
2026-09-01 15:49:03 | INFO | ============================================================
2026-09-01 15:49:03 | INFO | PHASE 1: GENERATE ZARR STORES
2026-09-01 15:49:03 | INFO | ============================================================
2026-09-01 15:49:03 | INFO | Subset 'frontal_structure': zarr stores complete (or exports already on disk) for all dates — skipping generate.
2026-09-01 15:49:03 | INFO | 
2026-09-01 15:49:03 | INFO | ====================================================

Co-locating 8 channels
[2011-12-04T00_00_00]
Read gradb2 with shape: (12960, 17280)
Processing config: D
Loading config from: /Users/jaketallman/PycharmProjects/front-finding/src/front_finding/finding/configs/finding_config_D.yaml
Thresholding with window size 64 and threshold 85 and mode pool


/Users/jaketallman/PycharmProjects/front-finding/.venv/lib/python3.12/site-packages/numpy/lib/_nanfunctions_impl.py:1593: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a,
/Users/jaketallman/PycharmProjects/front-finding/.venv/lib/python3.12/site-packages/numpy/lib/_nanfunctions_impl.py:1593: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a,
/Users/jaketallman/PycharmProjects/front-finding/.venv/lib/python3.12/site-packages/numpy/lib/_nanfunctions_impl.py:1593: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a,
/Users/jaketallman/PycharmProjects/front-finding/.venv/lib/python3.12/site-packages/numpy/lib/_nanfunctions_impl.py:1593: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a,
/Users/jaketallman/PycharmProjects/front-finding/.venv/lib/python3.12/site-packages/numpy/lib/_nanfunctions_impl.py:1593: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a,
/Users/jaketallman/PycharmProjects/front-finding/.venv/

## What it found

In [ ]:
def product(kind):
    """Path to one of this run's outputs."""
    return properties_io.get_global_front_output_path(
        OUT_DIR, TIME_STR, kind, RUN_TAG)


geometry = pd.read_parquet(product("geometry"))     # shape, one row per front
index = pd.read_parquet(product("front_index"))     # label -> name, bbox
props = pd.read_parquet(product("properties"))      # co-located channels

binary = np.load(finding_io.binary_filename(
    TIMESTAMP, cfg.finding.config, cfg.run_id), mmap_mode="r")

print(f"grid          {binary.shape[0]:,} x {binary.shape[1]:,}")
print(f"front pixels  {int(np.count_nonzero(binary)):,}")
print(f"fronts        {len(geometry):,}")
geometry.head()

In [ ]:
px = int(geometry["npix"].sum())
print(f"front pixels        {px:,}  ({px / binary.size:.3%} of the grid)")
print(f"total length        {geometry['length_km'].sum():,.0f} km")
print(f"  median / longest  {geometry['length_km'].median():.1f} / "
      f"{geometry['length_km'].max():,.0f} km")
print(f"pixels per front    median {geometry['npix'].median():.0f}, "
      f"max {geometry['npix'].max():,}")
print(f"branched            {(geometry['num_branches'] > 0).mean():.1%}")
print(f"latitude span       {geometry['centroid_lat'].min():.1f} to "
      f"{geometry['centroid_lat'].max():.1f}")

geometry[["npix", "length_km", "orientation",
          "num_branches", "mean_curvature"]].describe().T

Front length is heavy-tailed — mostly short filaments, a few basin-scale
structures — so the size axes are logged.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].hist(np.log10(geometry["length_km"].clip(lower=1e-3)), bins=60)
axes[0].set_xlabel("log10 length (km)")
axes[0].set_ylabel("fronts")

axes[1].hist(np.log10(geometry["npix"]), bins=60)
axes[1].set_xlabel("log10 pixels")

axes[2].hist(geometry["orientation"].dropna(), bins=60)
axes[2].set_xlabel("orientation (deg)")

fig.suptitle(f"{len(geometry):,} fronts — {TIMESTAMP}")
fig.tight_layout()

Centroids come from the geometry table, so this needs no part of the label
map. Colour is length: the long coherent structures stand out from the
filament background.

In [ ]:
fig, ax = plt.subplots(figsize=(13, 6))
s = ax.scatter(geometry["centroid_lon"], geometry["centroid_lat"],
               c=np.log10(geometry["length_km"].clip(lower=1e-3)),
               s=0.5, cmap="viridis")
ax.set_xlabel("longitude")
ax.set_ylabel("latitude")
ax.set_title(f"Front centroids — {TIMESTAMP}")
fig.colorbar(s, ax=ax, label="log10 length (km)")
fig.tight_layout()

The longest front, with the gradb2 it was found in. The bounding box comes
from the front index; the field is re-read from the store and cropped to that
window.

In [ ]:
longest = geometry.loc[geometry["length_km"].idxmax()]
row = index[index["label"] == longest["label"]].iloc[0]

pad = 50
y0, y1 = max(0, int(row.y0) - pad), int(row.y1) + pad
x0, x1 = max(0, int(row.x0) - pad), int(row.x1) + pad

field = llc_source.read_channel(
    cfg, TIMESTAMP, "gradb2", "frontal_structure")[y0:y1, x0:x1]
crop = np.asarray(binary[y0:y1, x0:x1])
shaded = np.log10(np.clip(field, 1e-20, None))

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5), sharex=True, sharey=True)
axes[0].imshow(shaded, origin="lower", cmap="magma", aspect="auto")
axes[0].set_title("log10 gradb2")

axes[1].imshow(shaded, origin="lower", cmap="gray", aspect="auto")
axes[1].imshow(np.ma.masked_where(~crop, crop), origin="lower",
               cmap="autumn", aspect="auto", interpolation="nearest")
axes[1].set_title("fronts found")

fig.suptitle(f"{longest['name']} — {longest['length_km']:,.0f} km, "
             f"{int(longest['npix']):,} px")
fig.tight_layout()

## The per-front table

`group` writes shape, `colocate` writes the sampled channels. Joined on the
front label, that is one feature row per front — the thing the dataset is
eventually built from.

In [ ]:
fronts = geometry.merge(props, left_on="label", right_on="flabel",
                        suffixes=("", "_prop"))
medians = [c for c in fronts.columns if c.endswith("_median")]

print(f"{len(fronts):,} fronts x {len(fronts.columns)} columns")
fronts[["name", "length_km"] + medians].head()

In [ ]:
# Do the strongest fronts look different from the rest?
strong = fronts["gradb2_median"] > fronts["gradb2_median"].quantile(0.9)

comparison = pd.DataFrame({
    "weakest 90%": fronts.loc[~strong, medians].median(),
    "strongest 10%": fronts.loc[strong, medians].median(),
})
comparison["ratio"] = comparison["strongest 10%"] / comparison["weakest 90%"]
comparison

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].scatter(np.log10(fronts["gradb2_median"].clip(lower=1e-30)),
                np.log10(fronts["length_km"].clip(lower=1e-3)),
                s=1, alpha=0.2)
axes[0].set_xlabel("log10 gradb2 (median)")
axes[0].set_ylabel("log10 length (km)")
axes[0].set_title("Strength vs length")

axes[1].hist(fronts["turner_angle_median"].dropna(), bins=60)
axes[1].set_xlabel("Turner angle (median, deg)")
axes[1].set_ylabel("fronts")
axes[1].set_title("Thermohaline character")

ratio = np.log10((fronts["gradsalt2_median"]
                  / fronts["gradtheta2_median"].clip(lower=1e-30)).clip(lower=1e-30))
axes[2].hist(ratio.replace([np.inf, -np.inf], np.nan).dropna(), bins=60)
axes[2].set_xlabel("log10 gradsalt2 / gradtheta2")
axes[2].set_title("Haline vs thermal")

fig.tight_layout()

## What was written

`build-fronts --steps push` would copy this to
`s3://dbof/test_globals_for_front_finding/test01/20111204_000000/Fronts/`.

In [ ]:
for f in sorted(Path(OUT_DIR).iterdir()):
    print(f"{f.stat().st_size / 1e6:9.1f} MB  {f.name}")

for meta in sorted(Path(RUN_ROOT).glob("*.meta")):
    print(f"\n{meta.name}\n")
    print(meta.read_text())